## 1. Data 

In [ ]:
import pandas as pd

In [ ]:
file_path = "listings_newyork.xlsx"
listing = pd.read_excel(file_path)

In [ ]:
import pandas as pd

In [ ]:
listing.head()

## *Observation

In [ ]:
newyork = listing.copy()

In [ ]:
newyork.shape #36111 observations, 79 columns

In [ ]:
newyork.columns

In [ ]:
#Check outliers and correlation of numerical values only

In [ ]:
# Select numeric columns only
numeric_cols = newyork.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
newyork[numeric_cols].boxplot(figsize=(20,10), rot=90)
plt.title('Boxplots for all numeric columns')
plt.show()

In [ ]:
import seaborn as sns

In [ ]:
sub = newyork[numeric_cols].copy()
sub = sub.drop(columns=['calendar_updated', 'scrape_id'], errors='ignore')

In [ ]:
corr1 = sub.corr()
corr1

In [ ]:
plt.figure(figsize=(18,18))
sns.heatmap(corr1, vmin=-1, vmax=1, center=0, annot=True, cmap="coolwarm", annot_kws={"fontsize":5})
plt.title("Correlation Heatmap")
plt.show()

## 2. Data cleansing

## *Column dropping decision

In [ ]:
#minimum nights and maximum nights: keep the main one only

In [ ]:
#availability: Keep only _30 and _365 as the mimimum values and maximum value for the range of availability

In [ ]:
#number of reviews, review scores rating, calculated host listings counts: keep the main one only

In [ ]:
#reviews_per_month: drop as it highly correlated to number_of_reviews

In [ ]:
#as property type and room type show overlapping values >> drop the former one as it contains too many texts, we will just focus on the general listing types only

In [ ]:
newyork["property_type"].value_counts()

In [ ]:
newyork["room_type"].value_counts()

==> Consider keeping only room_type, as property_type contains text, and it is redundant to room_type

In [ ]:
listing["host_listings_count"].isnull().sum()

In [ ]:
listing["calculated_host_listings_count"].isnull().sum()

### ***Decision: May consider dropping:
- Pure identifiers / URLs
- Descriptive text / unstructured info
- Time-related
- Metadata (rarely useful in cross-section)
- Duplicative / overly detailed geographic or host info
- Columns that are aggregations of others (redundant)



In [ ]:
newyork.columns

In [ ]:
newyork.drop(columns=['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 
                      'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about',  'host_response_time', 'host_thumbnail_url', 
                      'host_picture_url', 'host_neighbourhood', 'host_listings_count','host_verifications', 'host_has_profile_pic', 
                      'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'latitude', 'longitude', 
                      'property_type', 'bathrooms_text', 'amenities', 'minimum_minimum_nights', 'maximum_minimum_nights', 
                       'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated',
                      'availability_60', 'availability_90', 'calendar_last_scraped', 'number_of_reviews_ltm', 
                       'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d','estimated_revenue_l365d', 
                       'first_review', 'last_review', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
                       'review_scores_communication', 'review_scores_location', 'review_scores_value', 'license', 
                      'calculated_host_listings_count_entire_homes',
                       'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 
                      'reviews_per_month'], inplace=True)   

In [ ]:
newyork.shape 
#same number of obs, yet the #cols was reduced to 20

In [ ]:
newyork.columns #after cleansing

In [ ]:
newyork.isnull().values.any()

In [ ]:
newyork.isnull().sum()

In [ ]:
newyork["has_availability"].value_counts()

*has_availability used to indicate if a listing is active and available for booking. It's often used in data analysis, where t (true) means the listing is active and f (false) means it is not ==> drop obs that are not available on Airbnb

In [ ]:
newyork.dropna(subset=['has_availability'], inplace=True)

In [ ]:
newyork.isnull().sum() #after dropping NA in has_availability

In [ ]:
newyork.shape

### *Dependent variable: Price

In [ ]:
newyork["price"].describe() #original 'price' data

==> It is shown that the distribution of price is right-skewed with the median < mean. Also, it was observed that the 'price' variable contains extreme outliers, as the max is 50,104, while the 75th percentile is only 279. That’s >170× larger than typical prices

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["price"], bins=100, edgecolor='black')
plt.title("Distribution of Price")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
numeric_cols2 = newyork.select_dtypes(include=['float64', 'int64']).columns

In [ ]:
newyork[numeric_cols2].boxplot(figsize=(20,10), rot=90)
plt.title('Boxplots for all numeric columns')
plt.show()

In [ ]:
import numpy as np

In [ ]:
newyork["log_price"] = np.log(newyork["price"]+1)

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["log_price"], bins=100, edgecolor='black')
plt.title("Distribution of Log-Transformed Price")
plt.xlabel("log(price)")
plt.ylabel("Frequency")
plt.show()

==> Observation:
- The distribution of log_price is approximately normal (bell-shaped), which is more suitable for linear regression.

- The original price variable had extreme right skewness and many outliers, but the log transformation has reduced skewness and stabilized variance.

- A few high values remain, but they are not extreme anymore and do not require removal at this stage.

- Outliers will not be removed since they may represent real high-value Airbnb listings.

- Missing values in price (and therefore log_price) will be dropped before regression, because regression cannot handle missing target values.

In [ ]:
#Removing missing values in price

In [ ]:
newyork.dropna(subset=['price'], inplace=True)

In [ ]:
newyork["price"].isnull().values.any()

In [ ]:
newyork["price"].describe() 

### *Independent variables

In [ ]:
newyork.isnull().sum() #after dropping missing values in price

In [ ]:
#host_response_rate >> mean substitution

In [ ]:
newyork["host_response_rate"].describe()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["host_response_rate"], bins=20, edgecolor='black')
plt.title("Distribution of Host response rate")
plt.xlabel("Response rate")
plt.ylabel("Frequency")
plt.show()

In [ ]:
response_rate_mean = newyork["host_response_rate"].mean()
response_rate_mean

In [ ]:
newyork.fillna({"host_response_rate": response_rate_mean}, inplace = True)

In [ ]:
#host_acceptance_rate >> mean substitution

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["host_acceptance_rate"], bins=20, edgecolor='black')
plt.title("Distribution of Host acceptance rate")
plt.xlabel("Acceptance rate")
plt.ylabel("Frequency")
plt.show()

In [ ]:
acceptance_rate_mean = newyork["host_acceptance_rate"].mean()
acceptance_rate_mean

In [ ]:
newyork.fillna({"host_acceptance_rate": acceptance_rate_mean}, inplace = True)

In [ ]:
newyork.isnull().sum()

In [ ]:
#review_scores_rating >> mean substitution

In [ ]:
newyork["review_scores_rating"].describe()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["review_scores_rating"], bins=20, edgecolor='black')
plt.title("Distribution of Review Scores Rating")
plt.xlabel("Review Score")
plt.ylabel("Frequency")
plt.show()

In [ ]:
rv_score_mean = newyork["review_scores_rating"].mean()
rv_score_mean

In [ ]:
newyork.fillna({"review_scores_rating": rv_score_mean}, inplace = True)

In [ ]:
newyork.isnull().sum()

In [ ]:
newyork.head()

In [ ]:
#host is superhost >> mode substitute

In [ ]:
newyork["host_is_superhost"].value_counts()

In [ ]:
host_is_superhost_mode = newyork["host_is_superhost"].mode()[0]
host_is_superhost_mode

In [ ]:
newyork.fillna({"host_is_superhost": host_is_superhost_mode}, inplace = True)

In [ ]:
#host total listing count >>  mode substitute

In [ ]:
host_total_listings_count_mode = newyork["host_total_listings_count"].mode()[0]
host_total_listings_count_mode

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["host_total_listings_count"], bins=20, edgecolor='black')
plt.title("Distribution of Host total listings")
plt.xlabel("Number of listings")
plt.ylabel("Frequency")
plt.show()

In [ ]:
newyork.fillna({"host_total_listings_count": host_total_listings_count_mode}, inplace = True)

In [ ]:
#bathrooms  >> mode substitute

In [ ]:
newyork["bathrooms"].value_counts()

In [ ]:
bathrooms_mode = newyork["bathrooms"].mode()[0]
bathrooms_mode

In [ ]:
newyork.fillna({"bathrooms": bathrooms_mode}, inplace = True)

In [ ]:
#bedroom  >> mode substitute

In [ ]:
bedrooms_mode = newyork["bedrooms"].mode()[0]
bedrooms_mode

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["bedrooms"], bins=20, edgecolor='black')
plt.title("Distribution of Number of Bedrooms")
plt.xlabel("Number of Bedrooms")
plt.ylabel("Frequency")
plt.show()

In [ ]:
newyork.fillna({"bedrooms": bedrooms_mode}, inplace = True)

In [ ]:
newyork.isnull().sum()

In [ ]:
#bed >>  mode substitute

In [ ]:
newyork["beds"].value_counts()

In [ ]:
beds_mode = newyork["beds"].mode()[0]
beds_mode

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(newyork["beds"], bins=20, edgecolor='black')
plt.title("Distribution of Number of Beds")
plt.xlabel("Number of Beds")
plt.ylabel("Frequency")
plt.show()

In [ ]:
newyork.fillna({"beds": beds_mode}, inplace = True)

In [ ]:
newyork.isnull().sum() #all missing values were handled

In [ ]:
newyork.shape

## 3. Data transformation

### *Dummy variables

In [ ]:
newyork.head()

In [ ]:
newyork["host_is_superhost"].value_counts()

In [ ]:
newyork["neighbourhood_group_cleansed"].value_counts()

In [ ]:
newyork["room_type"].value_counts()

In [ ]:
newyork["instant_bookable"].value_counts()

In [ ]:
newyork2 = pd.get_dummies(newyork, columns=["host_is_superhost", "neighbourhood_group_cleansed", "room_type", "instant_bookable"],drop_first=True, dtype=int)

In [ ]:
newyork2.drop(columns=["has_availability"],inplace=True) #drop availability column for now as it contains only active listings

In [ ]:
newyork2.head()

In [ ]:
newyork2.shape

### *Multicollinearity check

In [ ]:
newyork2.columns

In [ ]:
corr2 = newyork2.corr()
corr2

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(corr2, vmin=-1, vmax=1, center=0, annot=True, cmap="coolwarm", annot_kws={"fontsize":7})
plt.title("Correlation Heatmap")
plt.show()

## 4. Initial Model

In [ ]:
import statsmodels.api as sm

In [ ]:
newyork2.columns.get_loc("log_price")

In [ ]:
y = newyork2.iloc[:,15]

In [ ]:
newyork2.shape

In [ ]:
newyork2.columns

In [ ]:
x = newyork2.iloc[:,[0,1,2,3,4,5,6,8,9,10,11,12,13,14,16,17,18,19,20,21,22,23,24]]

In [ ]:
x.columns

In [ ]:
x.shape

In [ ]:
x_c = sm.add_constant(x)

In [ ]:
reg1 = sm.OLS(y,x_c).fit()
print(reg1.summary())

### *Multicollinearity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif

In [ ]:
x.values

In [ ]:
x.shape[1]

In [ ]:
vif_factors = [vif(x.values,i) for i in range(0, x.shape[1])]

In [ ]:
pd.DataFrame({"Variable": x.columns, "VIF Factor": vif_factors})

#### **Correction

In [ ]:
x2 = newyork2.iloc[:,[1,2,3,4,5,6,8,9,10,11,12,13,14,16,17,18,19,20,21,22,23,24]] #drop host_response_rate due to higher VIF

In [ ]:
x2_c = sm.add_constant(x2)

In [ ]:
reg2 = sm.OLS(y,x2_c).fit()
print(reg2.summary())

In [ ]:
x2.values

In [ ]:
x2.shape[1]

In [ ]:
vif_factors_2 = [vif(x2.values,i) for i in range(0, x2.shape[1])]

In [ ]:
pd.DataFrame({"Variable": x2.columns, "VIF Factor": vif_factors_2})

In [ ]:
#drop host_acceptance_rate and review_scores_rating due to high VIF

In [ ]:
newyork2.columns.get_loc("review_scores_rating")

In [ ]:
x3 = newyork2.iloc[:,[2,3,4,5,6,8,9,10,11,12,14,16,17,18,19,20,21,22,23,24]]

In [ ]:
x3.head()

In [ ]:
x3.shape

In [ ]:
x3_c = sm.add_constant(x3)

In [ ]:
reg3 = sm.OLS(y,x3_c).fit()
print(reg3.summary())

In [ ]:
x3.values

In [ ]:
x3.shape[1]

In [ ]:
vif_factors_3 = [vif(x3.values,i) for i in range(0, x3.shape[1])]

In [ ]:
pd.DataFrame({"Variable": x3.columns, "VIF Factor": vif_factors_3})

==> Multicollinearity issue was fixed

### *Heteroscedasticity

In [ ]:
residual = reg3.resid

In [ ]:
exog = reg3.model.exog

In [ ]:
bp_test = sm.stats.het_breuschpagan(residual, exog)
bp_test

In [ ]:
labels = ["Test Statistics", "p value", "f statistics", "f p-values"]

In [ ]:
list(zip(labels, bp_test))

As the p-value and f-value are both smaller than 0.05, there is a heteroscedasticity issue detected

#### **Correction

In [ ]:
reg4 = sm.OLS(y,x3_c).fit(cov_type = "HC3")
print(reg4.summary())

### *Linearity

In [ ]:
#compare fitted vs. actual

In [ ]:
y_pred = reg4.predict() #model with constant term and heteroscedasticity issue fixed

In [ ]:
plt.figure(figsize=(4,4))
sns.regplot(x=y_pred, y=y)
plt.title("Actual vs Predicted Values (log_price)")
plt.xlabel("Predicted log_price")
plt.ylabel("Actual log_price")
plt.show()

In [ ]:
#compare residual vs. fitted

In [ ]:
residual = reg4.resid

In [ ]:
plt.figure(figsize=(4,4))
sns.regplot(x=y_pred, y=residual)
plt.title("Residual vs. Fitted values")
plt.xlabel("Fitted values (Predicted log_price)")
plt.ylabel("Residuals")
plt.show()

==> This suggests non-linearity and some model misspecification.

==> May need polynomial terms or interaction effects to better explain variance.

### *Normality of errors

In [ ]:
# Q-Q Plot (residual normality)

In [ ]:
plt.figure(figsize = (2,2))
sm.qqplot(residual, line = "s")
plt.title("Normality of errors")
plt.show()

### *If we replace missing values in 'price' by median(price)

In [ ]:
listing2 = listing.copy()

In [ ]:
listing2.head()

In [ ]:
listing2.drop(columns=['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name',
                      'host_since', 'host_location', 'host_about',  'host_response_time', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count','host_verifications', 'host_has_profile_pic', 
                      'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'latitude', 'longitude', 
                      'property_type', 'bathrooms_text', 'amenities', 'minimum_minimum_nights', 'maximum_minimum_nights', 
                       'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated',
                      'availability_60', 'availability_90', 'calendar_last_scraped', 'number_of_reviews_ltm', 
                       'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d','estimated_revenue_l365d', 
                       'first_review', 'last_review', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin',
                       'review_scores_communication', 'review_scores_location', 'review_scores_value', 'license', 'calculated_host_listings_count_entire_homes',
                       'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month'], inplace=True)   

In [ ]:
listing2.columns #after cleaning redundant and useful data

In [ ]:
listing2.isnull().sum()

In [ ]:
# has availability

In [ ]:
listing2.drop(columns=["has_availability"], inplace = True)

In [ ]:
listing2.isnull().sum()

In [ ]:
#price

In [ ]:
Q1 = listing2["price"].quantile(0.25)
Q3 = listing2["price"].quantile(0.75)

In [ ]:
IQR = Q3 - Q1

In [ ]:
lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

In [ ]:
listing3 = listing2[(listing2["price"] > lower) & (listing2["price"] < upper)]

In [ ]:
listing3["price"].describe() 

In [ ]:
price_med = listing3["price"].median()
price_med

In [ ]:
listing2.fillna({"price": price_med}, inplace = True)

In [ ]:
listing2["price"].describe()

In [ ]:
plt.hist(listing2["price"])
plt.show()

In [ ]:
listing2["log_price"] = np.log(listing2["price"]+1)

In [ ]:
listing2.columns

In [ ]:
listing2["log_price"]

In [ ]:
plt.hist(listing2["log_price"])
plt.show()

In [ ]:
listing2.isnull().sum()

In [ ]:
response_rate_mean1 = listing2["host_response_rate"].mean()
response_rate_mean1

In [ ]:
listing2.fillna({"host_response_rate": response_rate_mean1}, inplace = True)

In [ ]:
acceptance_rate_mean1 = listing2["host_acceptance_rate"].mean()
acceptance_rate_mean1

In [ ]:
listing2.fillna({"host_acceptance_rate": acceptance_rate_mean1}, inplace = True)

In [ ]:
rv_score_mean1 = listing2["review_scores_rating"].mean()
rv_score_mean1

In [ ]:
listing2.fillna({"review_scores_rating": rv_score_mean1}, inplace = True)

In [ ]:
host_is_superhost_mode1 = listing2["host_is_superhost"].mode()[0]
host_is_superhost_mode1

In [ ]:
listing2.fillna({"host_is_superhost": host_is_superhost_mode1}, inplace = True)

In [ ]:
host_total_listings_count_mode1 = listing2["host_total_listings_count"].mode()[0]
host_total_listings_count_mode1

In [ ]:
listing2.fillna({"host_total_listings_count": host_total_listings_count_mode1}, inplace = True)

In [ ]:
bathrooms_mode1 = listing2["bathrooms"].mode()[0]
bathrooms_mode1

In [ ]:
listing2.fillna({"bathrooms": bathrooms_mode1}, inplace = True)

In [ ]:
bedrooms_mode1 = listing2["bedrooms"].mode()[0]
bedrooms_mode1

In [ ]:
listing2.fillna({"bedrooms": bedrooms_mode1}, inplace = True)

In [ ]:
beds_mode1 = listing2["beds"].mode()[0]
beds_mode1

In [ ]:
listing2.fillna({"beds": beds_mode1}, inplace = True)

In [ ]:
listing2.isnull().sum()

In [ ]:
listing4 = pd.get_dummies(listing2, columns=["host_is_superhost", "neighbourhood_group_cleansed", "room_type", "instant_bookable"],drop_first=True, dtype=int)

In [ ]:
lt_corr = listing4.corr()
lt_corr

In [ ]:
#Correlation heatmap
plt.figure(figsize=(15,15))
sns.heatmap(lt_corr, vmin=-1, vmax=1, center=0, annot=True, cmap="coolwarm", annot_kws={"fontsize":7})
plt.show()

In [ ]:
#Regression

In [ ]:
listing4.columns.get_loc("log_price")

In [ ]:
y_lt4 = listing4.iloc[:,15]

In [ ]:
listing4.shape

In [ ]:
listing4.columns

In [ ]:
x_lt4 = listing4.iloc[:,[2,3,4,5,6,8,9,10,11,12,14,16,17,18,19,20,21,22,23,24]] 
#not include high VIF variables

In [ ]:
x_lt4.columns

In [ ]:
x_lt4.shape

In [ ]:
x_lt4_c = sm.add_constant(x_lt4)

In [ ]:
reg_lt4_1 = sm.OLS(y_lt4,x_lt4_c).fit(cov_type="HC3")
print(reg_lt4_1.summary())

In [ ]:
#comparision
print(reg4.summary())
#reg model when removing missing values in price shows better results

### 5. Final model

In [ ]:
print(reg4.summary()) 